# 04a - Trade-off Bias dan Varians
Materi Modul 4: pertukaran bias-varians (Bagian 1 & 2), regulasi dan pemilihan model.

- **Bias**: error karena model terlalu sederhana (underfitting)
- **Varians**: error karena model terlalu sensitif pada data latih (overfitting)
- Error total = Bias² + Varians + Noise

Cara melihatnya: latih model yang sama pada **banyak dataset berbeda** dari proses yang sama.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)

def f_asli(x):
    return np.sin(2 * np.pi * x)

x_uji = np.linspace(0, 1, 100)

## 1. Latih model pada 100 dataset berbeda

In [ ]:
n_dataset = 100
n_data = 40
noise = 0.3
derajat_list = [1, 4, 9]

prediksi = {d: [] for d in derajat_list}

for _ in range(n_dataset):
    x = rng.uniform(0, 1, n_data)
    y = f_asli(x) + rng.normal(0, noise, n_data)
    for d in derajat_list:
        model = make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression())
        model.fit(x.reshape(-1, 1), y)
        prediksi[d].append(model.predict(x_uji.reshape(-1, 1)))

for d in derajat_list:
    prediksi[d] = np.array(prediksi[d])
print({d: prediksi[d].shape for d in derajat_list})

## 2. Visualisasi
Garis tipis = tiap dataset. Garis merah = rata-rata prediksi. Garis hitam = fungsi asli.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
for a, d in zip(ax, derajat_list):
    for p in prediksi[d][:30]:
        a.plot(x_uji, p, color='steelblue', alpha=0.2)
    a.plot(x_uji, prediksi[d].mean(axis=0), color='red', label='Rata-rata prediksi')
    a.plot(x_uji, f_asli(x_uji), color='black', label='Fungsi asli')
    a.set_ylim(-2, 2); a.set_title(f'Derajat {d}')
ax[0].legend()
plt.tight_layout(); plt.show()

## 3. Hitung Bias² dan Varians

In [ ]:
hasil = []
for d in derajat_list:
    rata_pred = prediksi[d].mean(axis=0)
    bias2 = np.mean((rata_pred - f_asli(x_uji)) ** 2)
    varians = np.mean(prediksi[d].var(axis=0))
    hasil.append((d, bias2, varians, bias2 + varians))
    print(f'Derajat {d:2d} | Bias^2={bias2:.4f} | Varians={varians:.4f} | Bias^2+Varians={bias2 + varians:.4f}')

## 4. Kurva bias-varians untuk semua derajat

In [ ]:
semua_d = range(1, 11)
b2, var = [], []
rng = np.random.default_rng(1)
for d in semua_d:
    preds = []
    for _ in range(80):
        x = rng.uniform(0, 1, n_data)
        y = f_asli(x) + rng.normal(0, noise, n_data)
        m = make_pipeline(PolynomialFeatures(d, include_bias=False), StandardScaler(), LinearRegression()).fit(x.reshape(-1, 1), y)
        preds.append(m.predict(x_uji.reshape(-1, 1)))
    preds = np.array(preds)
    b2.append(np.mean((preds.mean(axis=0) - f_asli(x_uji)) ** 2))
    var.append(np.mean(preds.var(axis=0)))

b2, var = np.array(b2), np.array(var)
plt.plot(semua_d, b2, marker='o', label='Bias^2')
plt.plot(semua_d, var, marker='o', label='Varians')
plt.plot(semua_d, b2 + var, marker='o', label='Bias^2 + Varians')
plt.yscale('log')
plt.xlabel('Kompleksitas model (derajat)'); plt.legend()
plt.title('Trade-off bias-varians')
plt.show()
print('Derajat dengan total error terkecil:', list(semua_d)[int(np.argmin(b2 + var))])

## Catatan
- (hubungan kompleksitas model dengan bias dan varians, kenapa regularisasi membantu menurunkan varians, dll.)